# Alpamayo-v1.5 Gradient-Guided BFA Demo

Targets the **expert denoiser** and **action projections** in BF16.
Ranks bit-flip coordinates by a first-order loss estimate (`grad · Δw`),
then for each (bit, coord) measures the actual post-flip loss delta.

Output: `bfa_results.json` with per-trial `{bit, flat_idx, delta_loss, ...}`.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa        # sits next to this notebook

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to("cuda")
model.eval()
# We need requires_grad on target weights for bit-grad.
# (HF models default to requires_grad=True, so this is usually a no-op.)
for p in model.parameters():
    p.requires_grad_(True)

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
clip_id = clip_ids[774]   # same default as inference.ipynb
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    "cuda",
)

In [ ]:
# GT action in the same form the diffusion model is trained on.
gt_action = model.action_space.traj_to_action(
    traj_history_xyz=data["ego_history_xyz"].cuda().to(torch.float32),
    traj_history_rot=data["ego_history_rot"].cuda().to(torch.float32),
    traj_future_xyz=data["ego_future_xyz"].cuda().to(torch.float32),
    traj_future_rot=data["ego_future_rot"].cuda().to(torch.float32),
)
# action_space returns (B, 1, n_waypoints, 2); squeeze traj_group dim → (B, n_waypoints, 2)
gt_action = gt_action.view(-1, *model.action_space.get_action_space_dims())
print("gt_action:", gt_action.shape, gt_action.dtype)

with torch.autocast("cuda", dtype=torch.bfloat16):
    ctx = bfa.build_fm_context(model, model_inputs, gt_action)

# Fix a noise sample so every trial uses the SAME x_t for comparable losses.
torch.cuda.manual_seed_all(42)
fixed_noise = torch.randn_like(ctx.gt_action)
T_VAL = 0.5

def loss_fn():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        return bfa.fm_one_step_loss(model, ctx, t_val=T_VAL, noise=fixed_noise)

baseline = loss_fn().item()
print("baseline FM loss:", baseline)

In [ ]:
targets = bfa.collect_target_linears(model)
print(f"{len(targets)} target Linear modules")
print("first 5:", list(targets.keys())[:5])

# Single backward pass — grads are cloned out, so we can modify weights later.
t0 = time.time()
grads = bfa.compute_clean_grads(model, targets, loss_fn)
print(f"backward done in {time.time()-t0:.1f}s; "
      f"example grad norm: {list(grads.values())[0].norm().item():.4f}")

## Smoke tests

Lightweight checks. Run these BEFORE the main BFA loop — if any fails,
investigate before spending ~1 hour on the full sweep.

In [ ]:
# Cell 10: bit-flip primitives work as expected
w = torch.tensor([1.0], dtype=torch.bfloat16, device="cuda")

# sign flip: 1.0 → -1.0
assert bfa.bf16_xor_bit_all(w, 15).item() == -1.0

# exponent MSB flip: 1.0 has biased exp 0x7F = 01111111
#   flipping bit 14 → 11111111 → +Inf or NaN (non-finite)
flipped = bfa.bf16_xor_bit_all(w, 14)
assert not torch.isfinite(flipped).item()

# mantissa LSB flip: 1.0 → 1 + 2^-7
flipped = bfa.bf16_xor_bit_all(w, 0)
assert abs(flipped.float().item() - (1.0 + 2**-7)) < 1e-6

print("bit-flip correctness: OK")

In [ ]:
# Cell 11: flip-then-restore returns the weight tensor to its exact pre-flip state
mod = list(targets.values())[0]
snapshot = mod.weight.data.clone()

orig, flipped = bfa.bf16_flip_one(mod.weight.data, flat_idx=123, bit=14)
assert not torch.equal(mod.weight.data, snapshot), (
    "flip produced no change — is flat_idx=123 valid for this module?"
)
bfa.restore_one(mod.weight.data, 123, orig)
assert torch.equal(mod.weight.data, snapshot), "restore did not reproduce clean weight"

print("restore: OK")

In [ ]:
# Cell 12: baseline loss is deterministic across calls with fixed noise,
# and gradient-guided top-k is at least as strong as a random coord at the
# same bit. (Not a strict assertion — single-sample; we expect top-k to win
# on average in the full loop.)
with torch.no_grad():
    l1 = loss_fn().item()
    l2 = loss_fn().item()
assert abs(l1 - l2) < 1e-5, (l1, l2)
print(f"baseline determinism: OK ({l1})")

bit = 14   # exponent MSB — catastrophic bit
name = list(targets.keys())[0]
mod = targets[name]
flat_idx, _ = bfa.topk_bitflip_coords(mod.weight.data, grads[name], bit=bit, k=1)
r_top = bfa.measure_flipped_loss(model, ctx, mod, flat_idx[0].item(), bit, loss_fn)

rng = np.random.default_rng(0)
random_idx = int(rng.integers(0, mod.weight.numel()))
r_rand = bfa.measure_flipped_loss(model, ctx, mod, random_idx, bit, loss_fn)

print(f"top-k delta : {r_top['post_loss'] - baseline:.4g}")
print(f"random delta: {r_rand['post_loss'] - baseline:.4g}")

In [ ]:
BITS = list(range(16))      # full BF16 sweep. Start with [14, 15] for a quick smoke.
TOP_K_PER_MODULE = 20       # per (module, bit). Scale up after smoke test.
MODULE_SUBSET = list(targets.keys())   # or filter to a few names for speed

results = []
t_start = time.time()
for bit in BITS:
    for name in MODULE_SUBSET:
        mod = targets[name]
        w = mod.weight.data
        g = grads[name]
        flat_idx, bg_vals = bfa.topk_bitflip_coords(w, g, bit=bit, k=TOP_K_PER_MODULE)
        for i, idx in enumerate(flat_idx.tolist()):
            r = bfa.measure_flipped_loss(
                model, ctx, mod, flat_idx=idx, bit=bit, loss_fn=loss_fn,
            )
            r.update({
                "module": name,
                "rank": i,
                "predicted_bit_grad": float(bg_vals[i].item()),
                "baseline_loss": baseline,
                "delta_loss": r["post_loss"] - baseline,
            })
            results.append(r)
    print(f"bit {bit} done, {len(results)} trials, {time.time()-t_start:.0f}s elapsed")

out_path = Path("bfa_results.json")
out_path.write_text(json.dumps(results, indent=2))
print(f"saved {len(results)} trials to {out_path}")

In [ ]:
df = pd.DataFrame(results)
print(df.groupby("bit")["delta_loss"].describe()[["count","mean","max"]])

# Top-10 worst flips overall (finite post_loss only — bit-14 exponent flips often go inf)
df_finite = df[df["post_loss_finite"] == 1.0]
print("\nTop-10 single-bit flips by delta_loss (finite-loss only):")
print(df_finite.sort_values("delta_loss", ascending=False).head(10)[
    ["module","bit","flat_idx","orig_value","flipped_value","delta_loss"]
])
n_inf = int((df["post_loss_finite"] == 0.0).sum())
print(f"\n({n_inf} additional trials produced non-finite loss — see post_loss_finite column)")


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
df.boxplot(column="delta_loss", by="bit", ax=ax)
ax.set_yscale("symlog")
ax.set_title("Per-bit delta_loss distribution (BF16, gradient-guided top-k)")
ax.set_xlabel("bit index (0=mantissa LSB, 14=exp MSB, 15=sign)")
ax.set_ylabel("post_loss - baseline_loss")
plt.suptitle("")
plt.tight_layout()
plt.savefig("bfa_per_bit.png", dpi=120)
plt.show()